<a id="top"></a>
# House Sales: MLflow tracking e Model Registry (esercitazione – modulo 4)

Questa esercitazione ti guida nell'uso di **MLflow per il tracking e la gestione dei modelli**.

**Obiettivi**:
- Tracciare esperimenti con `autolog`
- Confrontare modelli nella MLflow UI
- Registrare il modello migliore nel Model Registry
- Assegnare alias `production`
- Verificare il funzionamento della webapp

**Prerequisito**: Esercitazione E3 completata (pipeline già definite).

<a id="indice"></a>
## Indice

- [Step 0: Import e setup iniziale](#step0)
- [Step 1: Configurazione MLflow e primo log](#step1)
- [Step 2: Log del secondo modello e confronto nella UI](#step2)
- [Step 3: Registrazione del modello migliore](#step3)
- [Step 4: Assegnazione alias Production](#step4)
- [Step 5: Test della webapp](#step5)
- [Extra (opzionale): Logging avanzato](#extra)


<a id="step0"></a>
## Step 0: Setup iniziale

**Import necessari**:
```python
import mlflow
import mlflow.sklearn
from sklearn.metrics import mean_squared_error, r2_score
# + i tuoi import da E3
```

**Task**:
1. Copia da E3 il codice per caricare dati e creare `X_train`, `X_test`, `y_train`, `y_test`
2. Ricrea le due pipeline: `lr_pipeline` e `rf_pipeline`

**Hint**: Puoi letteralmente copiare le celle da E3!

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing

# ========================================
# STEP 0 - TRAINING PIPELINE (copiata da E3)
# ========================================

# Imports


# --------------------------------------------------
# 1) Caricamento dati
# --------------------------------------------------
# Prova prima con il dataset "house_sales" (se presente su OpenML).
# Se non esiste, usa un fallback robusto con California Housing.
try:
    data = fetch_openml(name="house_sales", version=1, as_frame=True)
    df = data.frame
except Exception:
    try:
        data = fetch_openml(name="house_sales_regression", version=1, as_frame=True)
        df = data.frame
    except Exception:
        cal = fetch_california_housing(as_frame=True)
        df = pd.concat([cal.data, cal.target.rename("price")], axis=1)

# --------------------------------------------------
# 2) Identificazione target e feature
# --------------------------------------------------
# Cerca il nome della colonna target in una lista tipica; se non la trova,
# usa l'ultima colonna come target.
target_candidates = ["price", "SalePrice", "sale_price", "target", "medv", "y"]
target = next((c for c in target_candidates if c in df.columns), df.columns[-1])

X = df.drop(columns=[target])
y = df[target]

# Manteniamo solo le feature numeriche (più adatte a slider/webapp e ai modelli)
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

if len(numeric_cols) == 0:
    raise ValueError("Nessuna feature numerica trovata. Controlla il dataset.")

X = X[numeric_cols]

# --------------------------------------------------
# 3) Train/test split
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------------------------------------
# 4) Preprocessing numerico
# --------------------------------------------------
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# --------------------------------------------------
# 5) Definizione delle pipeline
# --------------------------------------------------
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", numeric_transformer),
        ("model", LinearRegression())
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", numeric_transformer),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42
        ))
    ]
)

# --------------------------------------------------
# 6) Fit dei modelli
# --------------------------------------------------
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# --------------------------------------------------


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['bedrooms','bathrooms','sqft_living',...,'long','sqft_living15', 'sqft_lot15']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is 

<a id="step1"></a>
## Step 1: Primo modello con MLflow

**Configurazione MLflow**:
```python
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")  # necessario con MLflow 3.x (backend su file)

mlflow.set_tracking_uri("file:../mlruns")  # stesso store di bank!
mlflow.set_experiment("house_sales_regression")
mlflow.sklearn.autolog()
```

**Task**: Traccia il primo modello (LinearRegression)
- Usa `mlflow.start_run(run_name="linear_regression")`
- Fai `.fit()` della pipeline
- Calcola RMSE e R² sul test
- Logga con `mlflow.log_metric()`

**Verifica**: dalla root del progetto avvia la UI con
```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns
```
e controlla l'esperimento su http://127.0.0.1:5000.

💡 Su macOS la porta 5000 può essere occupata (AirPlay Receiver): in quel caso aggiungi `--port 5001`.

In [2]:
import os
import mlflow

import mlflow.sklearn

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("house_sales_regression")
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="linear_regression"):
    lr_pipeline.fit(X_train, y_train)

    y_pred = lr_pipeline.predict(X_test)
    test_rmse = mean_squared_error(y_test, y_pred) ** 0.5
    test_r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    print(f"Test RMSE: {test_rmse:.2f}")
    print(f"Test R²: {test_r2:.4f}")

2026/10/08 10:00:04 INFO mlflow.tracking.fluent: Experiment with name 'house_sales_regression' does not exist. Creating a new experiment.
2026/10/08 10:00:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/1

Test RMSE: 212539.52
Test R²: 0.7012


In [ ]:
# Per accedere alla UI di MLflow, apri un terminale nella root del progetto ed esegui:
# MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri file:mlruns
# Poi apri http://127.0.0.1:5000 nel browser.

<a id="step2"></a>
## Step 2: Secondo modello e confronto

**Task**: Traccia il Random Forest seguendo lo stesso pattern dello Step 1
- Usa `run_name="random_forest"`
- Fit di `rf_pipeline`
- Calcola e logga le stesse metriche

**Confronto nella UI**:
- Aggiorna la pagina (F5)
- Seleziona entrambe le run (checkbox)
- Clicca "Compare"
- Quale modello ha RMSE più basso?

In [5]:
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

models = {
    "random_forest": rf_pipeline,
    "svm": Pipeline(
        steps=[
            ("preprocessor", numeric_transformer),
            ("model", SVR(C=100_000, epsilon=10_000)),
        ]
    ),
}

if XGBRegressor is not None:
    models["xgboost"] = Pipeline(
        steps=[
            ("preprocessor", numeric_transformer),
            ("model", XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
            )),
        ]
    )
else:
    print("XGBoost non è installato. Per aggiungerlo, esegui: %pip install xgboost")

for run_name, pipeline in models.items():
    with mlflow.start_run(run_name=run_name):
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        rmse = mean_squared_error(y_test, predictions) ** 0.5
        r2 = r2_score(y_test, predictions)

        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)

        print(f"{run_name}: RMSE={rmse:.2f}, R²={r2:.4f}")

2026/10/08 10:04:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/08 10:05:14 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.1

random_forest: RMSE=146435.06, R²=0.8582


2026/10/08 10:05:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/08 10:05:32 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.1

svm: RMSE=216011.87, R²=0.6913


2026/10/08 10:06:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/10/08 10:06:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/workspaces/AI-Course/.venv/lib/python3.1

xgboost: RMSE=130393.00, R²=0.8875


<a id="step3"></a>
## Step 3: Registrazione nel Model Registry

**Dalla MLflow UI**:
1. Identifica il modello migliore (ordina per `test_rmse`)
2. Clicca sulla run → sezione "Logged models" → clicca sul modello `model`
3. Clicca "Register model"
4. **Nome**: `house_sales_rf_prod` (esatto!)
5. Conferma

💡 In MLflow 3 il modello non compare più nella scheda "Artifacts" della run, ma nella sezione "Logged models".

**Verifica**: Menu "Models" → dovresti vedere `house_sales_rf_prod`

⚠️ Il nome deve essere esatto, la webapp lo cerca così!

<a id="step4"></a>
## Step 4: Alias Production

**Dalla MLflow UI**:
1. Menu "Models" → `house_sales_rf_prod`
2. Clicca su Version 1 → accanto ad "Aliases" clicca il pulsante per aggiungere un alias
3. Scrivi `production`
4. Conferma con "Save aliases"

**Verifica**: Dovresti vedere il tag "production" accanto alla versione.

💡 La webapp cerca `models:/house_sales_rf_prod@production`

<a id="step5"></a>
## Step 5: Test webapp

**Avvia la webapp**:
```bash
cd Webapp
uv run streamlit run app_house.py
```

**Verifica**:
- Si apre `http://localhost:8501`
- Nella sidebar: gli slider con le caratteristiche dell'abitazione
- Modifica gli slider → le predizioni cambiano
- Prova "Stima il prezzo di vendita"

**Troubleshooting**:
- Errore "Model not found" → controlla nome e alias
- Errore tracking URI → verifica `../mlruns`

✅ Se funziona, hai completato l'esercitazione!

In [ ]:
# ========================================
# STEP 5: TEST DA TERMINALE
# ========================================

# Esegui dal terminale:
# cd Webapp
# uv run streamlit run app_house.py

# Poi verifica nel browser che tutto funzioni!